# MECHA in metafspm, 8: the standard problem's boundaries, the osmotic scenarios and the flows by layer

MECHA's **standard problem** solves each section alone between the soil and its conduits; its uptake per potential
difference and root surface is kr. Its boundary conditions are parameters of the plant (`doc/equations.rst`, The
standard problem):

| boundary | parameters |
|---|---|
| the soil, around the section | `standard_psi_soil`, and `standard_psi_soil_right` for a soil wetter on one side |
| the xylem (barrier > 0) | its potential `standard_psi_xylem`, or its flow: `standard_axial_condition` 2, `standard_axial_flow` |
| the phloem | the protophloem's potential at barrier 0, `standard_psi_sieve`; the sieve tubes' at barrier > 0, `standard_psi_sieve_tube`; or the phloem's flow, `standard_sieve_flow` |
| the elongation zone (barrier 0) | the water the growing cells take: `standard_elongation` 1, `elongation_midpoint_rate`, `elongation_side_rate_difference` |
| the osmotic potentials | `standard_osmotic` 1 with MECHA's presets (MechaOsmoticPresets) |

Each is compared with the legacy's scenarios (`test_scenario_boundaries.py`, `test_boundary_conditions.py`,
`test_osmotic.py`, `test_transmembrane_fractions.py`). **Run time:** about a minute.

> **Developer tutorial 8** of ten:
> | | notebook | what it covers |
> |---|---|---|
> | 0 | `00_overview.ipynb` | the package's modules, the original MECHA and its reactions branch, how the port is validated, the docs |
> | 1 | `01_granap_mecha.ipynb` | GRANAP's anatomies and MECHA's hydraulics coupled in a plant model; MECHA's inputs by role; scenarios; the barriers |
> | 2 | `02_soil_gradient.ipynb` | the plant problem with a soil drier on one side of a root section; a root partly out of contact, by position or by cells |
> | 3 | `03_needle_solutes.ipynb` | a pine needle of three sections, its transpiration, MechaSolutes over a day |
> | 4 | `04_full_plant.ipynb` | a whole seedling in its soil, run to the fixed point |
> | 5 | `05_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |
> | 6 | `06_solute_storage_and_decay.ipynb` | the solutes' storage fractions, their first-order loss and the liquid boundary |
> | 7 | `07_section_options.ipynb` | the section's hydraulic options: the barrier codes 0-11, φ-thickenings, aquaporins, plasmodesmata rules, walls, xylem pieces, tracheids, measured axial conductances |
> | 8 | `08_standard_problem.ipynb` | the standard problem's boundaries (soil sides, xylem and phloem pressures or flows, the elongation zone), the osmotic scenarios, the flows by layer |
> | 9 | `09_solute_options.ipynb` | the solutes' pathways, schemes and time steps, held concentrations and steady states, the coupled water and sugar |
>
> New to MECHA? Start with the user tutorials (`../user/01_first_run.ipynb`): they solve a root section and explain
> its results without the framework's details.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openalea.metafspm.coupling.composite_wrapper import CompositeModel
from openalea.metafspm.coupling.translator import Translator
from openalea.metafspm.scene.scene import Scene

from openalea.granap.metafspm.architectures import RootAxis
from openalea.granap.metafspm.component import GranapAnatomy, junction_rules
from openalea.mecha.component import MechaHydraulics


class RootSection(CompositeModel):                     # the first tutorial's plant model
    initiators = (RootAxis, GranapAnatomy)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    def __init__(self, data_structure, time_step):
        self.anatomy = GranapAnatomy(data_structure=data_structure, time_step=time_step)
        self.hydraulics = MechaHydraulics(data_structure=data_structure, time_step=time_step)
        self.declare_data_and_couple_components(translator_path=Translator(),
                                                components=(self.anatomy, self.hydraulics))

    def run(self):
        self.anatomy()
        self.hydraulics()


def section(anatomy=None, **parameters):
    """A scene of one root section (GRANAP's *anatomy*: its parameters; the monocot root by default), MECHA's
    *parameters* in its scenario; returns (scene, its DataStructure, its MechaHydraulics)."""
    scenario = dict(parameters)
    if anatomy is not None:
        scenario["root_anatomy"] = anatomy
    scene = Scene(pd.DataFrame([dict(plant="root", model=RootSection, x=0., y=0., z=0., rotation=0.,
                                     scenario={"parameters": scenario})]))
    return scene, scene.populations[0].data_structure, scene.populations[0].instance.hydraulics


def solve(scene, barrier=1., **parameters):
    """One step at *barrier* with MECHA's *parameters* changed for it (then restored): the section's kr
    (cm hPa-1 d-1), Kx (cm4 hPa-1 d-1) and standard uptake (cm3 d-1)."""
    ds, mecha = scene.populations[0].data_structure, scene.populations[0].instance.hydraulics
    saved = {name: getattr(mecha, name) for name in parameters}
    for name, value in parameters.items():
        setattr(mecha, name, value)
    ds.set("barrier", np.full(ds.n_nodes(), float(barrier)))
    try:
        scene.run()
    finally:
        for name, value in saved.items():
            setattr(mecha, name, value)
    return pd.Series({"kr": float(ds.get("kr")[0]), "Kx": float(ds.get("Kx")[0]),
                      "uptake": float(ds.get("standard_uptake")[0])})

## 1. kr: the reference case

The soil at 0 hPa all around, the xylem at −5000 hPa, a Casparian strip (barrier 1):

In [2]:
scene, ds, mecha = section()
reference = solve(scene)
reference

kr        0.000110
Kx        1.194155
uptake    0.002772
dtype: float64

## 2. The soil: one side wetter

`standard_psi_soil` is the soil's potential on the left of the section (relative x 0), `standard_psi_soil_right` on
its right (x 1), linear in between. The uptake is each border node's own exchange; kr takes the soil's mean
potential at the nodes in contact:

In [3]:
sides = pd.DataFrame({"0 hPa all around": reference,
                      "−1000 left, 0 right": solve(scene, standard_psi_soil=-1000., standard_psi_soil_right=0.),
                      "−3000 left, 0 right": solve(scene, standard_psi_soil=-3000., standard_psi_soil_right=0.)}).T
sides

,kr,Kx,uptake
0 hPa all around,0.00011,1.194155,0.002772
"−1000 left, 0 right",0.00011,1.194155,0.002494
"−3000 left, 0 right",0.00011,1.194155,0.001939


kr barely changes: the section's conductances are linear, so the mean soil potential sets the uptake. Tutorial 2
shows where the water enters and leaves around the section.

## 3. The xylem: its potential or its flow

With a flow condition (`standard_axial_condition` 2), the flow `standard_axial_flow` (cm³ d⁻¹) leaves by the xylem,
split among the vessels as their areas; the xylem's potential is then their mean, and kr is computed with it. With
the reference's uptake as the flow, the same physical state:

In [4]:
flow = solve(scene, standard_axial_condition=2., standard_axial_flow=reference["uptake"])
pd.DataFrame({"potential −5000 hPa": reference, "flow, the same uptake": flow}).T

,kr,Kx,uptake
potential −5000 hPa,0.00011,1.194155,0.002772
"flow, the same uptake",0.00011,1.194155,0.002772


kr agrees to its fourth digit: each vessel takes its share of the flow, and their potentials barely spread.

## 4. The phloem and the elongation zone

At barrier 0 (the root tip) the xylem is not yet conducting: the water goes to the **protophloem**, and the growing
cells take some (the elongation sink: a wall's or a protoplast's volume × the elongation rate × its water fraction,
`water_fraction_apoplast`, `water_fraction_symplast`). GRANAP's roots have no protophloem cells (they tag no cell
`protophloem`), so here the tip's water comes from the soil only, for the elongation:

In [5]:
tip = solve(scene, 0., standard_elongation=1., elongation_midpoint_rate=2.8, elongation_side_rate_difference=0.6)
sink = ds.get("standard_elongation_sink")
print(f"uptake {tip['uptake']:.3g} cm3 d-1, the elongation's sink {sink.sum():.3g} cm3 d-1 "
      f"({(sink > 0).sum()} nodes; MECHA's 2.8 cm d-1, maize, Pritchard et al. 1988)")

uptake 0.00953 cm3 d-1, the elongation's sink 0.00953 cm3 d-1 (5503 nodes; MECHA's 2.8 cm d-1, maize, Pritchard et al. 1988)


The uptake equals the sink: the growing tissue draws its water from the soil. With a side difference (0.6 cm d⁻¹
faster on the right), the right side's cells grow faster and take more: the root bends (MECHA's hydrotropism
studies).

At barrier > 0 the **sieve tubes** can be given a potential (`standard_psi_sieve_tube`) or a flow
(`standard_sieve_flow`; negative: the phloem unloads water into the section):

In [6]:
phloem = pd.DataFrame({
    "reference": reference,
    "sieve tubes at +11000 hPa": solve(scene, standard_psi_sieve_tube=1.1E4),
    "the phloem unloading 2e-5 cm3 d-1": solve(scene, standard_sieve_flow=-2.0E-5),
}).T
phloem

,kr,Kx,uptake
reference,0.000110,1.194155,0.002772
sieve tubes at +11000 hPa,0.000103,1.194155,0.002593
the phloem unloading 2e-5 cm3 d-1,0.000110,1.194155,0.002771


Pressurized sieve tubes push water into the stele: the soil's uptake falls by 7 %. Most of the phloem's water goes to
the xylem next to it: of the 2 × 10⁻⁵ cm³ d⁻¹ unloaded, the soil's uptake falls by only about 5 %. kr, defined from
the soil's uptake, is then no longer the section's conductivity alone: compare kr only between cases with the same
phloem condition.

## 5. The osmotic scenarios

The standard problem's kr (scenario 0) has no osmotic terms. With `standard_osmotic` 1, each membrane adds σ Δπ to
the potential difference across it, π the cells' and the walls' osmotic potentials, σ its reflection coefficient.
MECHA's presets (`MechaOsmoticPresets`) give them:

- `os_hetero`: the cells' π, 0 `os_cortex` everywhere, 1–2 the legacy's profiles, 3 the cortex's gradient, 4 one
  value per tissue (`osmotic_epi`, `osmotic_exo`, `osmotic_endo_cell`, `osmotic_peri`, `osmotic_stele`);
- `s_hetero`: σ, 0 uniform, 1 the stele's halved, 2 the cortex's halved, × `s_factor`;
- the walls' solutions: the soil's (`osmotic_left_soil`, `osmotic_right_soil`, `osmotic_symmetry_soil` 1 left to
  right or 2 radial with `osmotic_shape_soil`) and the xylem's (`osmotic_xyl`, `osmotic_endo`,
  `osmotic_symmetry_xyl`).

The presets are a component of the plant model, before MECHA:

In [7]:
from openalea.mecha.osmotic import MechaOsmoticPresets


class RootWithPresets(RootSection):
    def __init__(self, data_structure, time_step):
        self.anatomy = GranapAnatomy(data_structure=data_structure, time_step=time_step)
        self.osmotic = MechaOsmoticPresets(data_structure=data_structure, time_step=time_step)
        self.hydraulics = MechaHydraulics(data_structure=data_structure, time_step=time_step)
        self.declare_data_and_couple_components(translator_path=Translator(),
                                                components=(self.anatomy, self.osmotic, self.hydraulics))

    def run(self):
        self.anatomy()
        self.osmotic()
        self.hydraulics()


def osmotic(**presets):
    """A section with MECHA's osmotic presets *presets*, solved with the osmotic terms: its kr and uptake."""
    scenario = dict(presets, standard_osmotic=1.)
    scene = Scene(pd.DataFrame([dict(plant="root", model=RootWithPresets, x=0., y=0., z=0., rotation=0.,
                                     scenario={"parameters": scenario})]))
    return solve(scene)


cases = {"no osmotic terms": dict(os_hetero=0., s_factor=0.),
         "os_hetero 0 (the cortex's −4800 hPa)": dict(os_hetero=0.),
         "os_hetero 1": dict(os_hetero=1.), "os_hetero 2": dict(os_hetero=2.),
         "os_hetero 0, the stele's σ halved": dict(os_hetero=0., s_hetero=1.),
         "a saline soil (−2000 hPa)": dict(os_hetero=0., osmotic_left_soil=-2000., osmotic_right_soil=-2000.),
         "soil −300 left, −900 right": dict(os_hetero=0., osmotic_left_soil=-300., osmotic_right_soil=-900.,
                                            osmotic_symmetry_soil=1.)}
osmotic_table = pd.DataFrame({name: osmotic(**presets) for name, presets in cases.items()}).T
osmotic_table

,kr,Kx,uptake
no osmotic terms,0.000110,1.194155,0.002772
os_hetero 0 (the cortex's −4800 hPa),0.000139,1.194155,0.003489
os_hetero 1,0.000159,1.194155,0.004015
os_hetero 2,0.000160,1.194155,0.004022
"os_hetero 0, the stele's σ halved",0.000172,1.194155,0.004322
a saline soil (−2000 hPa),0.000106,1.194155,0.002658
"soil −300 left, −900 right",0.000136,1.194155,0.003434


**Reading it.** The cells' osmotic potential pulls water into them: with the same pressures the uptake rises. A
saline soil lowers the walls' potential outside the barrier and the uptake falls, as a drier soil would. The
reflection coefficients set how much of each difference acts.

## 6. Where the water crosses: the flows by layer

`standard_transmembrane_fraction` is each membrane's flow into its cell over the section's uptake; their sums by
layer row (`postprocess.transmembrane_layers`) and the axial flows into the xylem by layer (`axial_layers`, MECHA's
`flow_xyl_layer`) show where the water crosses membranes and where it reaches the vessels:

In [8]:
from openalea.mecha.postprocess import axial_layers, transmembrane_layers

solve(scene)
segment = int(ds.entity_ids("SubOrgan")[0])
layers = transmembrane_layers(ds, segment).set_index("layer")
xylem = axial_layers(mecha, segment).set_index("layer")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
layers[["fraction_plus", "fraction_minus"]].plot.bar(ax=axes[0], stacked=True, rot=0,
                                                     title="transmembrane fractions by layer row")
(-xylem["xylem"] * 1e6).plot.bar(ax=axes[1], rot=0, title="water leaving by the xylem, by its vessels' row")
axes[1].set_ylabel("10⁻⁶ cm³ d⁻¹")
plt.show()
print(f"fractions in: {layers['fraction_plus'].sum():.3f}, out: {layers['fraction_minus'].sum():.3f}")

fractions in: 1.325, out: -1.325


/tmp/ipykernel_1733860/1291280769.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The rows run from the stele (0) to the epidermis. The fractions in sum to more than 1 (1.3 here) and those out to
−1.3: part of the water crosses membranes twice, into a cell and out again (the endodermis', which a Casparian strip
makes every drop cross, and the cortex cells' on the way). The xylem's rows are its vessels' ranks.

## Try it yourself

1. At barrier 3 (suberized endodermis), where do the transmembrane fractions go?
2. Find the soil's osmotic potential that stops the uptake.
3. Give the tip's protophloem a potential: retag a few stele cells' tissue `protophloem` after the anatomy (as the
   sclerenchyma in tutorial 7) and set `standard_psi_sieve`.